# Predict Customer Churn
## Score: .91408

In [57]:
N_FOLDS = 5
N_SEEDS = 5
OPTUNA_TRIALS = 80
USE_RF_ET = False
USE_ORIGINAL_DATA = True
ORIGINAL_WEIGHT = 0.3
TARGET_ENC_M = 20
BLEND_MODE = 'rank'
USE_ISOTONIC_CALIBRATION = True
USE_CV_TARGET_ENCODING = True
STACK_RANK_LINEAR = False
USE_FEATURE_CLIP = False
USE_LOGREG_BLEND = False
USE_HGB = False
USE_DUAL_TRACK = True

In [58]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [ORIGINAL_WEIGHT] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [59]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    df['log_tenure'] = np.log1p(df['tenure'])
    df['TenureLow'] = (df['tenure'] < 12).astype(int)
    df['TenureMid'] = ((df['tenure'] >= 12) & (df['tenure'] < 36)).astype(int)
    df['TenureHigh'] = (df['tenure'] >= 36).astype(int)
    high_risk = (df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check') & (df['InternetService'] == 'Fiber optic')
    df['HighRiskCombo'] = high_risk.astype(int)
    df['FiberNoAddons'] = ((df['InternetService'] == 'Fiber optic') & (df['NumAddons'] == 0)).astype(int)
    df['MTM_ShortTenure'] = ((df['Contract'] == 'Month-to-month') & (df['tenure'] < 6)).astype(int)
    denom = 1 + df['NumAddons'] + df['NumStreaming']
    df['ChargePerService'] = df['MonthlyCharges'] / denom.replace(0, 1)
    df['ValueScore'] = (df['NumAddons'] + df['NumStreaming']) / (df['MonthlyCharges'] / 20 + 0.5)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

if USE_CV_TARGET_ENCODING:
    from sklearn.model_selection import StratifiedKFold
    cv_te = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
        X_full[f'{col}_te'] = np.nan
        for train_idx, val_idx in cv_te.split(X_full, y_full):
            tr = X_full.iloc[train_idx].join(y_full.iloc[train_idx])
            global_mean = tr['Churn'].mean()
            agg = tr.groupby(col)['Churn'].agg(['mean', 'count'])
            smooth = (agg['mean'] * agg['count'] + global_mean * TARGET_ENC_M) / (agg['count'] + TARGET_ENC_M)
            X_full.loc[val_idx, f'{col}_te'] = X_full.loc[val_idx, col].map(smooth).fillna(global_mean).values
        comb = pd.DataFrame({col: X_full[col], '_y': y_full.values})
        agg_full = comb.groupby(col)['_y'].agg(['mean', 'count'])
        smooth_full = (agg_full['mean'] * agg_full['count'] + y_full.mean() * TARGET_ENC_M) / (agg_full['count'] + TARGET_ENC_M)
        X_test_raw[f'{col}_te'] = X_test_raw[col].map(smooth_full).fillna(y_full.mean())
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])
else:
    for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
        X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

n_comp = len(train)
if USE_DUAL_TRACK and USE_ORIGINAL_DATA:
    X_comp_raw = engineer_features(train.drop(columns=['id', 'Churn']))
    tc_comp = X_comp_raw['TotalCharges'].median()
    X_test_comp_raw = engineer_features(test.drop(columns=['id']), total_charges_median=tc_comp)
    y_comp = train['Churn'].map({'Yes': 1, 'No': 0})
    if USE_CV_TARGET_ENCODING:
        for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
            X_comp_raw[f'{col}_te'] = np.nan
            for tr_idx, val_idx in cv_te.split(X_comp_raw, y_comp):
                tr = X_comp_raw.iloc[tr_idx].join(y_comp.iloc[tr_idx])
                gm = tr['Churn'].mean()
                agg = tr.groupby(col)['Churn'].agg(['mean', 'count'])
                smooth = (agg['mean'] * agg['count'] + gm * TARGET_ENC_M) / (agg['count'] + TARGET_ENC_M)
                X_comp_raw.loc[val_idx, f'{col}_te'] = X_comp_raw.loc[val_idx, col].map(smooth).fillna(gm).values
            comb = pd.DataFrame({col: X_comp_raw[col], '_y': y_comp.values})
            agg_full = comb.groupby(col)['_y'].agg(['mean', 'count'])
            smooth_full = (agg_full['mean'] * agg_full['count'] + y_comp.mean() * TARGET_ENC_M) / (agg_full['count'] + TARGET_ENC_M)
            X_test_comp_raw[f'{col}_te'] = X_test_comp_raw[col].map(smooth_full).fillna(y_comp.mean())
            X_comp_raw = X_comp_raw.drop(columns=[col])
            X_test_comp_raw = X_test_comp_raw.drop(columns=[col])
    else:
        for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
            X_comp_raw, X_test_comp_raw = target_encode(X_comp_raw, X_test_comp_raw, col, y_comp, m=TARGET_ENC_M)
            X_comp_raw = X_comp_raw.drop(columns=[col])
            X_test_comp_raw = X_test_comp_raw.drop(columns=[col])
    X_comp_encoded = pd.get_dummies(X_comp_raw, drop_first=True)
    X_test_comp_encoded = pd.get_dummies(X_test_comp_raw, drop_first=True)
    X_test_comp_encoded = X_test_comp_encoded.reindex(columns=X_comp_encoded.columns, fill_value=0)
else:
    X_comp_encoded = None
    X_test_comp_encoded = None
    y_comp = None

if USE_FEATURE_CLIP:
    num_cols = X_encoded.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        lo, hi = X_encoded[c].quantile(0.01), X_encoded[c].quantile(0.99)
        X_encoded[c] = X_encoded[c].clip(lo, hi)
        X_test_encoded[c] = X_test_encoded[c].clip(lo, hi)

In [60]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [61]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-12 01:49:10,249] A new study created in memory with name: no-name-3fad8a8f-787f-4077-961d-6b92ab4f21f4


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-12 01:50:12,337] Trial 0 finished with value: 0.9141115001380609 and parameters: {'n_estimators': 767, 'max_depth': 3, 'learning_rate': 0.029874531137255904, 'subsample': 0.6339320538140938, 'colsample_bytree': 0.9176844406194056, 'scale_pos_weight': 1.1376202199944643, 'reg_alpha': 0.04337620797302154, 'reg_lambda': 0.11588136288545971, 'min_child_weight': 9}. Best is trial 0 with value: 0.9141115001380609.
[I 2026-03-12 01:52:02,488] Trial 1 finished with value: 0.9121915490619749 and parameters: {'n_estimators': 617, 'max_depth': 7, 'learning_rate': 0.14118721726864714, 'subsample': 0.8805773259566565, 'colsample_bytree': 0.618309575357458, 'scale_pos_weight': 2.8514249941303067, 'reg_alpha': 1.336129713044157, 'reg_lambda': 0.5450302214139552, 'min_child_weight': 5}. Best is trial 0 with value: 0.9141115001380609.
[I 2026-03-12 01:53:04,402] Trial 2 finished with value: 0.9120335254768308 and parameters: {'n_estimators': 405, 'max_depth': 8, 'learning_rate': 0.1244058633

In [62]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-12 03:10:02,119] A new study created in memory with name: no-name-d1e65261-1d38-47c4-a0c9-91f4b4b2a642


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-12 03:10:46,237] Trial 0 finished with value: 0.9150215744939281 and parameters: {'n_estimators': 708, 'max_depth': 3, 'learning_rate': 0.06220959376559081, 'subsample': 0.6352020618259168, 'colsample_bytree': 0.6127174337894412, 'scale_pos_weight': 3.3633157757144976, 'reg_alpha': 0.5343086776628755, 'reg_lambda': 1.8781683821396553, 'min_child_samples': 49}. Best is trial 0 with value: 0.9150215744939281.
[I 2026-03-12 03:11:47,496] Trial 1 finished with value: 0.9144274471329392 and parameters: {'n_estimators': 769, 'max_depth': 8, 'learning_rate': 0.13778139864780636, 'subsample': 0.6744580585803647, 'colsample_bytree': 0.650780724141708, 'scale_pos_weight': 2.133435988046444, 'reg_alpha': 0.035852535164313956, 'reg_lambda': 0.0036693568050762283, 'min_child_samples': 70}. Best is trial 0 with value: 0.9150215744939281.
[I 2026-03-12 03:12:22,318] Trial 2 finished with value: 0.9143456209059074 and parameters: {'n_estimators': 525, 'max_depth': 3, 'learning_rate': 0.0442

In [63]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-12 04:10:55,238] A new study created in memory with name: no-name-f160f71c-9ff6-4396-b8e5-413fd09bd6c9


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-12 04:12:13,664] Trial 0 finished with value: 0.9140831773945891 and parameters: {'iterations': 523, 'depth': 3, 'learning_rate': 0.06069931626318953, 'subsample': 0.6565672531046075, 'colsample_bylevel': 0.7472057151274999, 'scale_pos_weight': 2.2133501842317216, 'l2_leaf_reg': 0.001520551411857836}. Best is trial 0 with value: 0.9140831773945891.
[I 2026-03-12 04:13:57,471] Trial 1 finished with value: 0.9143411823292384 and parameters: {'iterations': 356, 'depth': 8, 'learning_rate': 0.08680295918473017, 'subsample': 0.7396125904320957, 'colsample_bylevel': 0.6976515833276987, 'scale_pos_weight': 3.0260117007101743, 'l2_leaf_reg': 0.0059906462250618915}. Best is trial 1 with value: 0.9143411823292384.
[I 2026-03-12 04:16:18,628] Trial 2 finished with value: 0.9132644370482172 and parameters: {'iterations': 490, 'depth': 8, 'learning_rate': 0.14735104026231857, 'subsample': 0.7291616440632808, 'colsample_bylevel': 0.6897893126524718, 'scale_pos_weight': 2.540980091810211, 

In [64]:
if USE_HGB:
    from sklearn.ensemble import HistGradientBoostingClassifier

    def hgb_objective(trial):
        params = {
            'max_iter': trial.suggest_int('max_iter', 200, 600),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
            'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        }
        scores = []
        for train_idx, val_idx in cv.split(X_encoded, y_full):
            X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
            y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
            sw_tr = sample_weights[train_idx]
            m = HistGradientBoostingClassifier(**params, random_state=42)
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        return np.mean(scores)

    study_hgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
    study_hgb.optimize(hgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_hgb_params = study_hgb.best_params
    print(f"HGB Best CV AUC: {study_hgb.best_value:.4f}")
else:
    best_hgb_params = {}

In [65]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize
from scipy.stats import rankdata

n_models = (4 if USE_HGB else 3) if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))
oof_comp = np.zeros(n_comp) if (USE_DUAL_TRACK and X_comp_encoded is not None) else None
test_comp = np.zeros(len(X_test_encoded)) if (USE_DUAL_TRACK and X_comp_encoded is not None) else None

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_HGB:
        models.append(HistGradientBoostingClassifier(**best_hgb_params, random_state=seed+fold))
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    if oof_comp is not None:
        comp_tr = train_idx[train_idx < n_comp]
        comp_val = val_idx[val_idx < n_comp]
        if len(comp_tr) > 0 and len(comp_val) > 0:
            m_comp = LGBMClassifier(**best_lgb_params, random_state=42+fold, verbose=-1)
            m_comp.fit(X_comp_encoded.iloc[comp_tr], y_comp.iloc[comp_tr])
            oof_comp[comp_val] = m_comp.predict_proba(X_comp_encoded.iloc[comp_val])[:, 1]
            test_comp += m_comp.predict_proba(X_test_comp_encoded)[:, 1] / N_FOLDS

def to_rank_percentiles(arr):
    return (rankdata(arr) - 0.5) / len(arr)

oof_rank = np.column_stack([to_rank_percentiles(oof[:, i]) for i in range(n_models)])
oof_linear = oof.copy()

def neg_auc(w):
    blend = oof_blend_input @ w
    return -roc_auc_score(y_full, blend)

if STACK_RANK_LINEAR:
    oof_blend_input = oof_rank
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_rank = best_w
    oof_blend_input = oof_linear
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_linear = best_w
    blend_rank_oof = oof_rank @ w_rank
    blend_linear_oof = oof_linear @ w_linear
    best_auc, best_alpha = -1, 0.5
    for a in np.linspace(0, 1, 21):
        blended = a * blend_rank_oof + (1 - a) * blend_linear_oof
        auc = roc_auc_score(y_full, blended)
        if auc > best_auc:
            best_auc, best_alpha = auc, a
    blend_oof = best_alpha * blend_rank_oof + (1 - best_alpha) * blend_linear_oof
    blend_weights = best_alpha * w_rank + (1 - best_alpha) * w_linear
    stack_alpha = best_alpha
else:
    if BLEND_MODE == 'rank':
        oof_blend_input = oof_rank
    else:
        oof_blend_input = oof.copy()
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    blend_weights = best_w
    blend_oof = oof_blend_input @ blend_weights
    stack_alpha = None

dual_track_w = 1.0
if oof_comp is not None and USE_DUAL_TRACK:
    comp_idx = np.arange(n_comp)
    best_auc, best_w = -1, 1.0
    for w in np.linspace(0, 1, 21):
        blend_dual = w * blend_oof[comp_idx] + (1 - w) * oof_comp
        auc = roc_auc_score(y_comp, blend_dual)
        if auc > best_auc:
            best_auc, best_w = auc, w
    dual_track_w = best_w
    blend_oof = np.concatenate([dual_track_w * blend_oof[comp_idx] + (1 - dual_track_w) * oof_comp, blend_oof[n_comp:]])

use_logreg_blend = False
logreg_blender = None
if USE_LOGREG_BLEND and not STACK_RANK_LINEAR:
    from sklearn.linear_model import LogisticRegression
    logreg_blender = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    logreg_blender.fit(oof_rank, y_full)
    logreg_oof = logreg_blender.predict_proba(oof_rank)[:, 1]
    auc_scipy = roc_auc_score(y_full, blend_oof)
    auc_logreg = roc_auc_score(y_full, logreg_oof)
    if auc_logreg > auc_scipy:
        use_logreg_blend = True
        blend_oof = logreg_oof

if USE_ISOTONIC_CALIBRATION:
    iso_calib = IsotonicRegression(out_of_bounds='clip')
    iso_calib.fit(blend_oof, y_full)
    blend_oof = np.clip(iso_calib.predict(blend_oof), 0, 1)

names = ['XGB','LGB','Cat'] + (['HGB'] if USE_HGB else []) + (['RF','ET'] if USE_RF_ET else [])
mode_str = 'rank+linear' if STACK_RANK_LINEAR else ('logreg' if use_logreg_blend else BLEND_MODE)
print(f"Blend OOF AUC ({mode_str}): {roc_auc_score(y_full, blend_oof):.4f}")
if not use_logreg_blend:
    print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")
if oof_comp is not None and USE_DUAL_TRACK:
    print(f"Dual track (main weight): {dual_track_w:.3f}")
if STACK_RANK_LINEAR:
    print(f"Stack alpha (rank): {stack_alpha:.3f}")

Blend OOF AUC (rank): 0.9159
Weights: {'XGB': np.float64(0.333), 'LGB': np.float64(0.3336), 'Cat': np.float64(0.3334)}
Dual track (main weight): 0.400


In [66]:
def blend_test(tp, apply_cal=True):
    if use_logreg_blend:
        tp_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = logreg_blender.predict_proba(tp_rank)[:, 1]
    elif STACK_RANK_LINEAR:
        blend_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)]) @ w_rank
        blend_linear = tp @ w_linear
        preds = stack_alpha * blend_rank + (1 - stack_alpha) * blend_linear
    elif BLEND_MODE == 'rank':
        tp_input = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = tp_input @ blend_weights
    else:
        preds = tp @ blend_weights
    if apply_cal and USE_ISOTONIC_CALIBRATION:
        preds = np.clip(iso_calib.predict(preds), 0, 1)
    return preds

store_raw = (oof_comp is not None and USE_DUAL_TRACK)
all_test_preds = [blend_test(test_preds, apply_cal=not store_raw)]
all_test_comp = [test_comp.copy()] if test_comp is not None else None
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    tc = np.zeros(len(X_test_encoded)) if test_comp is not None else None
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
        if tc is not None:
            comp_tr = train_idx[train_idx < n_comp]
            comp_val = val_idx[val_idx < n_comp]
            if len(comp_tr) > 0 and len(comp_val) > 0:
                m_comp = LGBMClassifier(**best_lgb_params, random_state=base_seed+fold, verbose=-1)
                m_comp.fit(X_comp_encoded.iloc[comp_tr], y_comp.iloc[comp_tr])
                tc += m_comp.predict_proba(X_test_comp_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(blend_test(tp, apply_cal=not store_raw))
    if all_test_comp is not None and tc is not None:
        all_test_comp.append(tc)

if all_test_comp is not None and store_raw:
    final_main_raw = np.mean(all_test_preds, axis=0)
    final_comp = np.mean(all_test_comp, axis=0)
    blend_raw = dual_track_w * final_main_raw + (1 - dual_track_w) * final_comp
    final_preds = np.clip(iso_calib.predict(blend_raw), 0, 1) if USE_ISOTONIC_CALIBRATION else blend_raw
else:
    final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.070587
1,594195,0.001011
2,594196,0.092639
3,594197,0.002447
4,594198,0.515800
